# Group project: which cell types change in fly models of Parkinsonism?

**ISN Neurochemistry School 2026**

Parkinson's disease is defined by the loss of dopaminergic neurons. But patients
notice something else earlier: about a decade before any tremor, most of them
lose their sense of smell. Which cells are affected before the dying starts is
hard to study in people, because by the time a patient reaches a clinic it has
been going on for years.

Flies make it easy to look early. Pech et al. (*eLife*, 2025) built five fly
lines, each carrying a mutation in the fly version of a human Parkinson's gene,
and sequenced every cell in the brains of young flies. For each cell type they
counted how many genes changed, and compared that with how many you would expect
given how many cells of that type they captured.

So the measurement is a count of changed genes per cell type, at one age. Worth
keeping in mind as you go: it tells you where expression has shifted, which is
not the same as where cells are damaged or dying.

## The five models

| model | human gene | what the protein normally does |
|---|---|---|
| `LRRK2G2019S` | *LRRK2* | a kinase; the commonest genetic cause of Parkinson's |
| `PINK1P399L` | *PINK1* | tags damaged mitochondria so the cell can clear them |
| `Rab39BG192R` | *RAB39B* | moves membrane cargo around the cell |
| `AuxR927G` | *DNAJC6*, auxilin | recycles vesicles at the synapse |
| `SynjR258Q` | *SYNJ1*, synaptojanin | recycles vesicles at the synapse |

Start with whichever one interests you, then come back and run it again for
another. They do not all give the same answer, and the differences are worth
more than any single result.

**Questions you can ask**

1. Which cell types change most in the model you picked?
2. How solid is that? What is it built on?
3. Do the dopaminergic neurons, the ones that die in the disease, show anything?
4. Which genes change, and what do they have in common?
5. Are those genes specific to the affected cells, or does every neuron have them?
6. Do the five models agree with each other?

You do not have to do all of them. Two questions answered properly make a better
talk than six skimmed.

Everything is `pandas` and `scanpy`. No model, no training. About 30 minutes of
code, and then the interesting part.

## Running on Google Colab

Run the cell below first. On Colab it installs the software and fetches the
course files; on a laptop it does nothing. Run it either way.

In [ ]:
# Colab setup. On a laptop install this cell does nothing.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os

    !pip install -q scanpy openpyxl "pandas==2.2.3"

    # Get the course files, or update them if this session already has them.
    import subprocess

    if os.path.isdir("/content/course/.git"):
        update = subprocess.run(
            ["git", "-C", "/content/course", "pull", "--quiet", "--ff-only"],
            capture_output=True, text=True,
        )
        if update.returncode:
            print("Could not update the course files. Continuing with the copy")
            print("already in this session, which is almost certainly fine.")
    else:
        !rm -rf /content/course
        !git clone -q https://github.com/nkempynck/isn-school-2026 /content/course
    %cd /content/course
    print("Ready.")
else:
    print("Running locally, nothing to install.")

Running locally, nothing to install.


## 0. Setup, and your group's model

Pick a model and put it in `MY_MODEL`. Everything below follows from it, so
this is the only line you have to change. Come back and change it to compare.

In [ ]:
MY_MODEL = "AuxR927G"     # <-- pick one of the five. Change this first.

MODELS = ["LRRK2G2019S", "PINK1P399L", "Rab39BG192R", "AuxR927G", "SynjR258Q"]
assert MY_MODEL in MODELS, f"pick one of {MODELS}"

import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.set_figure_params(figsize=(6, 5), dpi=90, frameon=False)
print("working on:", MY_MODEL)

working on: AuxR927G


## 1. The data

For every cell type in the brain the authors worked out how much change in gene
expression you would **expect**, given how many cells of that type they
captured, then measured how far the real data missed that expectation.

That miss is the number in the table.

- **Large and positive**: this cell type is far more deregulated than its size
  explains. This is what we are looking for.
- **Around zero**: nothing unusual.
- **Large and negative**: it changed *less* than expected. This happens to the
  very large populations and is a property of the measure, not a finding.

In [ ]:
from scripts.download_data import pech_tables

paths = pech_tables()

cells = pd.read_excel(paths["celltypes"], sheet_name="normalized residuals z_scores")
cells = (cells[["cell type"] + MODELS]
         .dropna(subset=["cell type"])
         .set_index("cell type")
         .apply(pd.to_numeric, errors="coerce")
         .dropna(how="all"))

# Not every cluster was given a cell type name; the rest are still numbered.
named = cells[[not str(i).isdigit() for i in cells.index]]

print(f"\n{len(cells)} clusters, of which {len(named)} have a cell type name")
cells.head()

Published spreadsheets are laid out for human eyes, not for `pandas`. This one
has blank spacer columns and summary columns mixed in with the data, which is
why the cell above picks its columns by name instead of taking the sheet as it
comes. Expect to do this every time you use a supplementary table.

## 2. Question 1: which cell types does your model put into trouble?

Run this, then look at the figure before reading anything else.

In [ ]:
mine = named[MY_MODEL].sort_values(ascending=False).head(20).sort_values()

fig, ax = plt.subplots(figsize=(7.2, 0.3 * len(mine) + 1.3))
ax.barh(range(len(mine)), mine.values,
        color=["#c0392b" if v > 0 else "#7d9bc1" for v in mine.values])
ax.set_yticks(range(len(mine)))
ax.set_yticklabels(mine.index, fontsize=9)
ax.axvline(0, color="#333333", lw=0.8)
ax.grid(False)
ax.set_xlabel("deregulation, further right means further from normal")
ax.set_title(f"{MY_MODEL}: the twenty cell types that drift furthest", fontsize=11)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()

named[MY_MODEL].sort_values(ascending=False).head(5).round(1)

Answer these before moving on. Write the answers down; they go on your slides.

- Which cell type is top for your model, and how far clear of the second is it?
- **What is that cell type and what does it do?** If the abbreviation means
  nothing to you, look it up now. It is your whole result.
- Does your model have one standout, or a handful of similar ones?

## 3. Question 2: how solid is that?

Before presenting a number it is worth knowing what it rests on. This is the
same thing you would do to your own data, and it usually sharpens the result
rather than weakening it: it tells you which part of the story to lead with.

**How many cells is it built on?** A cluster seen 60 times in a mutant is not
the same evidence as one seen 600 times. There are also three different control
genotypes here, which is worth a look.

In [ ]:
counts = pd.read_excel(paths["counts"], sheet_name="Cell Counts").set_index("Unnamed: 0")
counts.index.name = "cluster"

MUTANTS = ["AuxR927G", "hRab39BG192R", "SynjR258Q", "hPink1P399L", "hLRRK2G2019S"]
CONTROLS = ["cantonized w1118 (control)", "DGRP-551", "w1118"]

my_top = named[MY_MODEL].idxmax()
counts.loc[[my_top], MUTANTS + CONTROLS]

**What does the significance test add?** The authors computed a confidence
interval per cell type and listed which ones fall outside it. Look up your cell
type, and then look at how many cell types pass in total, which tells you how
much the test is actually separating.

In [ ]:
ci = pd.read_excel(paths["celltypes"], sheet_name="Cell types outside CIs").set_index("celltype")

print("your top cell type, called outside the confidence interval?")
print(ci.loc[[my_top]].to_string())

print(f"\nout of {len(ci)} cell types, this many pass in each model:")
print(ci.sum(numeric_only=True).astype(int).to_string() if ci.select_dtypes('number').shape[1]
      else ci.apply(lambda c: (c == True).sum()).to_string())

**Do the tables line up?** The three sheets you have loaded all describe the
same clusters. Comparing the names they use is a thirty-second check that is
worth doing before you build anything on one of them.

In [ ]:
A = {str(i) for i in cells.index if not str(i).isdigit()}        # z-score sheet
B = {str(i) for i in ci.index if not str(i).isdigit()}           # confidence intervals
C = {str(i) for i in counts.index if not str(i).isdigit()}       # cell counts

print(f"named clusters: z-scores {len(A)}, confidence intervals {len(B)}, cell counts {len(C)}")
print("in the z-score sheet but not in the cell counts:", sorted(A - C))
print("in the cell counts but not in the z-score sheet:", len(C - A), "clusters")

**What was the design?** Look up the experiment on GEO (accession GSE228843):
how many genotypes, how many replicates, and how many ages. Ages matter here. If
you want to say one cell type is affected *before* another, think about what
that would take.

Now write down, in a sentence each:

- what your result supports
- what it does not reach to
- the one experiment you would do next

## 4. Question 3: what about the dopaminergic neurons?

Parkinson's disease kills dopaminergic neurons. This brain has them, annotated
as central brain dopaminergic neurons, `CB DAN`.

Compare them with your top cell type.

In [ ]:
WATCH = ["CB DAN", named[MY_MODEL].idxmax()]

cells.loc[WATCH, MODELS].round(1)

- What is the dopaminergic number in **your** model?
- Put it next to your top cell type. What do you conclude, and what can you
  *not* conclude from flies this young?

## 5. Is your result yours alone, or do the others agree?

Change `MY_MODEL` at the top and re-run to see another one, or read the five
columns side by side here.

In [ ]:
top = named.loc[named[MY_MODEL].sort_values(ascending=False).head(10).index]

fig, ax = plt.subplots(figsize=(7, 4.4))
sns.heatmap(top, cmap="RdBu_r", center=0, vmin=-200, vmax=200,
            annot=True, fmt=".0f", annot_kws={"size": 7},
            cbar_kws={"label": "deregulation"}, ax=ax)
ax.set_title(f"Your top ten cell types, in all five models", fontsize=11)
ax.set_ylabel("")
ax.grid(False)
plt.show()

# How similarly do the five models rank the cell types?
named.corr(method="spearman")[MY_MODEL].round(2).sort_values(ascending=False)

## 6. Question 4: which genes change in your model?

The authors ranked every fly gene by how deregulated it is, separately for each
model. **Rank 0 is the most deregulated**, so small numbers matter.

In [ ]:
genes = pd.read_excel(paths["genes"], sheet_name="Ranked_Genes_Fly")

# The gene names are in a column pandas calls "Unnamed: 10", because
# "sorted by median" in the sheet is a heading over the block, not a column.
cols = ["Unnamed: 10"] + [m + ".1" for m in MODELS] + ["median rank"]
genes = genes[cols].dropna(subset=["Unnamed: 10"])
genes.columns = ["gene"] + MODELS + ["median_rank"]
genes = genes.set_index("gene").apply(pd.to_numeric, errors="coerce").dropna()

print(f"{len(genes):,} genes ranked\n")
genes[MY_MODEL].nsmallest(15).astype(int)

Look up several of your top genes. Do they have anything in common? Does it fit
what your disease gene normally does, from the table at the top of this
notebook?

Be sceptical as well. Ribosomal RNA, mitochondrial transcripts and histone genes
turn up near the top of a lot of rankings for technical reasons. If your list is
full of them, say so: that is a finding about the experiment.

## 7. Question 5: are those genes special to the affected cells?

If your model's genes were only made by the cell type that got into trouble,
that would be a tidy explanation. Check it in the Fly Cell Atlas, the same file
you used in Practical 5.

In [ ]:
from scripts.prepare_data import make_fca_brain_h5ad

make_fca_brain_h5ad()          # 44 MB, from the course mirror
adata = sc.read_h5ad(ROOT / "data" / "processed" / "fca_brain.h5ad")

# The atlas names cell types by anatomy. Olfactory projection neurons are
# "antennal lobe projection neuron" here.
TYPES = ["antennal lobe projection neuron", "dopaminergic PAM neuron",
         "gamma Kenyon cell", "columnar neuron T1", "ensheathing glial cell"]

# Your model's top genes, keeping the ones the atlas measured.
MY_GENES = [g for g in genes[MY_MODEL].nsmallest(25).index if g in adata.var_names][:8]
print("showing:", MY_GENES)

sc.pl.dotplot(adata[adata.obs["annotation"].isin(TYPES)],
              var_names=MY_GENES, groupby="annotation")

- Are your genes specific to the cell type that was affected, or general?
- If every neuron makes them, why would one cell type fail first? Write down
  your best guess. You will not be able to prove it here, and saying so is part
  of the answer.

## 8. Your results

Run this. It prints what goes on your slides.

In [ ]:
mine_cells = named[MY_MODEL].sort_values(ascending=False)
mine_genes = genes[MY_MODEL].nsmallest(8).index.tolist()
agree = named.corr(method="spearman")[MY_MODEL].drop(MY_MODEL).sort_values(ascending=False)

print(f"MODEL                 {MY_MODEL}")
print(f"most affected         {mine_cells.index[0]}  ({mine_cells.iloc[0]:.0f})")
print(f"runner-up             {mine_cells.index[1]}  ({mine_cells.iloc[1]:.0f})")
print(f"dopaminergic neurons  CB DAN  ({cells.loc['CB DAN', MY_MODEL]:.1f})")
print(f"top genes             {', '.join(mine_genes)}")
print(f"agrees most with      {agree.index[0]}  (rho {agree.iloc[0]:.2f})")
print(f"agrees least with     {agree.index[-1]}  (rho {agree.iloc[-1]:.2f})")
print()
print("and the things that qualify it, from section 3:")
print(f"  cells behind it     {int(counts.loc[my_top, MUTANTS].min())} to "
      f"{int(counts.loc[my_top, MUTANTS].max())} per mutant")
print(f"  cell types passing  {int((ci == True).sum().max())} of {len(ci)} "
      f"pass the significance test, so passing it means little")
print(f"  ages sequenced      one (day 5) for four of the five models")

## What to present

Three slides, about five minutes.

1. **Your gene.** Which human Parkinson's gene you had, what the protein
   normally does, and what the experiment was: young flies, whole brain, before
   anything has degenerated.
2. **Your answer.** Your figure from section 2, with your top cell type and the
   dopaminergic number beside it. Then your genes from section 5 and what they
   have in common.
3. **How far it goes.** What your result supports, where it stops, and the one
   experiment you would do next. Use what you found in section 3: how many cells
   it rests on, what the significance test adds, and what a single timepoint can
   show.

That third slide is what makes it a talk. Anyone can read the top of a sorted
table; saying how far it reaches, and what you would measure next, is the part
people learn from.

If you looked at more than one model, say how they differed. If you looked at
one, say which and why.